![Platform+](images/banner.png)

# LAB 6: MARKETPLACE
## Build on data you don't own — a zero-copy view, no pipeline

> **Persona track: Data Engineer** — same lab, your angle.

:point_right: **In this lesson, you'll access a Private Listing shared with your account, then build a curated view on top of it — giving downstream models a clean interface without copying or loading a single row.**

The scenario: Roastery & Co. is opening a reading corner in their cafés. A fictional publisher — **Alpine Peaks Publishing** — has shared their forthcoming catalog privately with your Snowflake Education account. Rather than let every downstream model query the raw shared table, you'll expose a **curated view** in your own database: filtered to what the café needs, stable if the provider reshapes their schema, and still zero-copy — a view stores no rows.

We'll cover:
1. Two ways to get data: loading vs. querying in place
2. Public Marketplace vs. Private Listing — and when each fits
3. Searching the public Marketplace — by hand and with CoCo's skills
4. Getting access to a privately shared database
5. Inspecting the shared catalog
6. Your angle — building a zero-copy view over the share for downstream use

:books: **By the end you'll be able to:**
- Explain what "zero copy" means and when it's the right choice
- Distinguish Public Marketplace listings from Private Listings
- Search the public Marketplace by hand and with CoCo's `/marketplace-search` and `/get-marketplace-listing-details` skills
- Grant your role access to a privately shared database
- Build a view over a shared table that curates it for downstream consumers — without duplicating storage

**Time:** ~15 min · **Lab interface:** Workspace Notebook with Cortex Code (CoCo) as your tutor.
*Source data lives in the shared, read-only `PLATPLUS_DB` and in the `ALPINE_PEAKS_PUBLISHING` listing shared privately with this account.*

---
👉 **Connecting to your compute service:**
1. In the top bar, if no service is connected, click to connect and select your service.
2. Set your **role and warehouse** using the picker at the top-right — or just run the setup cell below, which sets them for you in SQL. (SQL cells run on the warehouse.)
3. Once the status shows **Connected** (green), run a cell with **Shift+Enter** (run + advance) or **Cmd/Ctrl+Enter** (run this cell only); the **▶** button works too.

> 📌 The service is a container that runs the notebook kernel and your **Python** cells. **SQL cells (and Snowpark pushdown) run on the query warehouse**, not the container. It stays running until it hits its idle timeout or you suspend it manually.
> 💡 **Best practice: Start a fresh chat for this lab.** Before you begin, open a fresh CoCo chat — click **New chat** (the **+** at the top of the panel). Each lab creates its own objects, so a fresh chat ensures CoCo isn't referencing tables or schemas from a prior lab that no longer exist. Shorter conversations also produce sharper answers and use fewer tokens.

In [ ]:
from snowflake.snowpark.context import get_active_session
session = get_active_session()
user = session.get_current_user().strip('"')
my_db = user + '_DB'
my_wh = user + '_WH'
my_role = user + '_RL'
print('Context ready. Signed in as ' + user)
print('Your workspace -> role: ' + my_role + ' | wh: ' + my_wh)

## Set Up Your Workspace
Set your **personal context** — your own role and warehouse — and point reads at the shared course data.

In [ ]:
%%sql -r dataframe_1
-- Set your personal context: your role + your own warehouse
USE ROLE {{my_role}};
CREATE WAREHOUSE IF NOT EXISTS {{my_wh}}
  WAREHOUSE_SIZE = XSMALL AUTO_SUSPEND = 60 AUTO_RESUME = TRUE INITIALLY_SUSPENDED = TRUE;
USE WAREHOUSE {{my_wh}};
-- Read from the shared, read-only course data
USE DATABASE PLATPLUS_DB;
USE SCHEMA ROASTERY_SALES;

## 1 · Two Ways to Get Data

Previously you queried data already in Snowflake. Today's question is *where data comes from in the first place*. It arrives in three shapes:

| Shape | Looks like | Roastery & Co. example |
|---|---|---|
| **Structured** | Rows + columns, fixed schema | `ORDERS`, `STORES` |
| **Semi-structured** | Flexible schema (JSON, Parquet, XML) — stored in a [`VARIANT`](https://docs.snowflake.com/en/user-guide/semistructured-intro) column | Loyalty app events |
| **Unstructured** | No schema at all | Supplier invoice PDFs, product photos |

Before worrying about shape, there's a more fundamental question: **do you need to bring it in at all?**

| | Load it in | Query in place (Marketplace / Sharing) |
|---|---|---|
| **You manage** | A pipeline + storage | Nothing — the provider maintains it |
| **Freshness** | As fresh as your last load | Always provider-current |
| **Best for** | Your own source files | Third-party / partner / reference data |

A publisher's book catalog is a good "query in place" case: the publisher maintains it, you just need to read it. The [**Snowflake Marketplace**](https://docs.snowflake.com/en/user-guide/data-marketplace-intro) and **Private Listings** let you access that data as a live, **zero-copy** share — it appears as a database in your account, but no data is copied and you pay no storage for it.

> :bulb: **The mindset shift:** loading (Lab 7) is one option, not the only one. For data a partner already maintains — a publisher's catalog, a supplier's price list, a vendor's reference feed — mounting a share is far less work than building and babysitting a pipeline.

> ⚙️ **Why this matters for engineers:** Zero-copy sharing eliminates an entire class of pipeline work. No ingestion jobs, no scheduling refreshes, no schema drift to manage. When a provider updates their data, your downstream tables and dynamic tables see it on the next refresh — automatically. That's fewer moving parts in your architecture and fewer 3 AM alerts.

## 2 · Public Marketplace vs. Private Listing

Snowflake offers two flavors of zero-copy data sharing:

| | Public Marketplace | Private Listing |
|---|---|---|
| **Who can see it** | Anyone on Snowflake | A curated list of invited accounts |
| **How to find it** | Marketplace (top-level) | Data sharing → External sharing → Shared with you |
| **Typical use** | Weather, demographics, financial data | Partner catalogs, internal divisions, supplier feeds |
| **Access** | Click **Get** in the UI | Provider shares directly; you grant access via SQL |

Today's listing — **Alpine Peaks Publishing** — is a **Private Listing**. Alpine Peaks Publishing (a fictional publisher) has shared their forthcoming catalog privately with the Snowflake Education account. You won't find it on the public Marketplace; it lives in your account's **Shared With You** tab.

:point_right: **Navigate to the listing in a new browser tab** (keep this notebook open):
1. In the Snowsight left navigation, **right-click → Open in new tab** on **Data sharing → External sharing** (so you keep this notebook open).
2. Click the **Shared With You** tab.
3. Find and click the **Alpine Peaks Publishing** listing and review its description.

> 💡 **Already provisioned — just browse.** You'll likely see a **"Get Access"** button — that's expected. In a real scenario, clicking it sends a request to your account administrator to grant you access to this shared database. In this lab, Section 4 below runs a stored procedure that acts as the administrator on your behalf and grants your role the necessary privileges. Once that's done, the button changes to **Open**. For now, just review the listing description and note what's included.

> 📚 [Snowflake Marketplace overview](https://docs.snowflake.com/en/user-guide/data-marketplace) · [Private data sharing](https://docs.snowflake.com/en/user-guide/data-sharing-intro)

---

![Alpine Peaks Publishing listing](images/ob_pde_alpine_peaks_listing_1_v2.png)

## 3 · Discover Data on the Public Marketplace

You've seen the *concept* of public vs. private sharing. Now see how you'd actually **find** data on the public **Snowflake Marketplace** — the open catalog any Snowflake account can browse.

**Search it yourself in Snowsight** (open a new browser tab, keep this notebook open):
1. In the left navigation, **right-click → Open in new tab** on **Snowflake Marketplace** (you'll see three options: Snowflake Marketplace, Consumer Hub, and Provider Studio — choose **Snowflake Marketplace**).
2. Use the **search bar** at the top to describe what you need — e.g. `coffee`, `foot traffic`, `consumer spending`.
3. Narrow results with the **Provider**, **Category**, and **Business Need** filters on the left.
4. Click any listing to read its description and sample data. For a free listing, **Get** adds it to your account as a zero-copy share.

> :bulb: As an engineer, skim a listing for **fit**: how often is it refreshed, is the schema stable, and would you query it in place or ingest it? Public listings shine for broad **reference data** — weather, demographics, points of interest. Docs: [Exploring listings on the Marketplace](https://docs.snowflake.com/en/user-guide/data-marketplace-intro).

### Let CoCo drive the search. 🤖
You don't have to click through the UI — CoCo ships two Marketplace skills. Before using them, ask CoCo to explain how they differ (this is a plain question, so **don't** prefix it with a slash — a slash would launch a skill instead of answering):

🤖 **CoCo Prompt** — *click to copy, then paste into the CoCo panel:*

<pre style="user-select: all; -webkit-user-select: all; background: rgba(41, 181, 232, 0.12); border-left: 4px solid #29B5E8; padding: 12px; border-radius: 0 4px 4px 0; cursor: pointer; white-space: pre-wrap; word-wrap: break-word;">I see two Marketplace skills: /marketplace-search and /get-marketplace-listing-details. How do they differ, and when should I use each one?</pre>

> 🔎 In short: **`/marketplace-search`** casts a wide net and returns a *list* of matching listings; **`/get-marketplace-listing-details`** zooms in on *one* listing you name and explains what's in it and whether it fits your need.

### Search the Marketplace with CoCo. 🤖
Now run a real search for something Roastery-adjacent:

🤖 **CoCo Prompt** — *click to copy, then paste into the CoCo panel:*

<pre style="user-select: all; -webkit-user-select: all; background: rgba(41, 181, 232, 0.12); border-left: 4px solid #29B5E8; padding: 12px; border-radius: 0 4px 4px 0; cursor: pointer; white-space: pre-wrap; word-wrap: break-word;">/marketplace-search Are there any coffee-related listings available?</pre>

CoCo returns a short list of coffee- and café-adjacent listings, each with a title, a one-line description, and a link. Skim them and notice how varied third-party data can be.

### Zoom in on one listing. 🤖
Pick a listing from those results and get the full picture with the second skill:

🤖 **CoCo Prompt** — *click to copy, then paste into the CoCo panel:*

<pre style="user-select: all; -webkit-user-select: all; background: rgba(41, 181, 232, 0.12); border-left: 4px solid #29B5E8; padding: 12px; border-radius: 0 4px 4px 0; cursor: pointer; white-space: pre-wrap; word-wrap: break-word;">/get-marketplace-listing-details All Coffee Shop Locations in the Top 5 US Cities</pre>

CoCo returns the listing's provider, what data it contains, and how it might (or might not) fit Roastery & Co.'s needs.

> :point_right: **So why aren't we using a public listing today?** Public listings are perfect for broad reference data — but Roastery needs one specific thing: **Alpine Peaks Publishing's forthcoming catalog**, shared *directly and privately* with this account (the listing you reviewed earlier). You won't surface it by searching the public Marketplace — it's private to us. It's the closest fit for the job, so let's grant access and build a curated view over it.

## 4 · Get Access to the Shared Database

The `ALPINE_PEAKS_PUBLISHING` database already exists in this account — the instructor mounted it once. What each learner needs is **read access** to it: a non-owner role can't query a shared database until it's granted privileges on the objects inside.

A stored procedure owned by a privileged role handles this in one call, so you don't need `ACCOUNTADMIN`:

:point_right: Run the cell below to grant your `_RL` role access to the shared database.

In [ ]:
%%sql -r dataframe_2
-- Grant your role access to the shared database (one-time, per user)
CALL PLATPLUS_DB.RESOURCES.ACCESS_ALPINE_PEAKS_PUBLISHING_DATA('{{my_role}}');

> 🔎 **What just happened.** The procedure granted your role **`IMPORTED PRIVILEGES`** on the shared database — the standard consumer-side way to open a database created from a share. When you inspect your grants you'll see them surface as targeted rows — `USAGE` on the database, `USAGE` on the `BOOKS` schema, `SELECT` on the `FORTHCOMING_RELEASES` table — because that's exactly what the **provider** chose to put in the share. You can read that one table, nothing more.

> :bulb: **How you open a share — `IMPORTED PRIVILEGES`.** A consumer can't hand out object-level grants on a shared database — Snowflake blocks `GRANT SELECT` on shared objects. The one lever is `GRANT IMPORTED PRIVILEGES ON DATABASE … TO ROLE`, which passes through exactly the objects the **provider** placed in the share (here, one table). Want finer control? The **provider** can define **database roles** to segment the share (Alpine Peaks hasn't), or you can build a **view** over the shared table in your own database and grant on that. The provider controls *what's* in the share; you control *who* in your account can reach it.

> ⚙️ **Under the hood:** This procedure runs with **owner's rights** (ACCOUNTADMIN) — it grants your role access to the shared database on your behalf. You'll see this pattern often in managed environments: a privileged procedure self-serves access so engineers don't need to file tickets or wait for an admin.

## 5 · Inspect the Shared Database

A shared database behaves like any other: it has schemas and tables you can explore with `SHOW` and `SELECT`. Run the three cells below to get oriented before you query.

In [ ]:
%%sql -r dataframe_3
-- What schemas are in the shared database?
SHOW SCHEMAS IN DATABASE ALPINE_PEAKS_PUBLISHING;

In [ ]:
%%sql -r dataframe_4
-- What tables are available?
SHOW TABLES IN DATABASE ALPINE_PEAKS_PUBLISHING;

In [ ]:
%%sql -r dataframe_5
-- Preview the forthcoming releases catalog
SELECT *
FROM ALPINE_PEAKS_PUBLISHING.BOOKS.FORTHCOMING_RELEASES
LIMIT 10;

> 🔎 **Note the columns.** The table has `TITLE`, `AUTHOR`, `CATEGORY`, and `RELEASE_DATE` — a clean structured catalog of upcoming book releases. No joins needed to start exploring.

## 6 · Build a Curated View for Downstream

The café-planning models shouldn't all query the raw shared table directly — if the publisher reshapes their schema, every model breaks. Your job as the engineer: expose a **stable, curated view** in your own database. Start broad to understand the data, then build the interface.

In [ ]:
%%sql -r dataframe_6
-- How many titles are in the catalog?
SELECT COUNT(*)
FROM ALPINE_PEAKS_PUBLISHING.BOOKS.FORTHCOMING_RELEASES;

In [ ]:
%%sql -r dataframe_7
-- What genres does this publisher cover?
SELECT DISTINCT CATEGORY
FROM ALPINE_PEAKS_PUBLISHING.BOOKS.FORTHCOMING_RELEASES
ORDER BY 1;

In [ ]:
%%sql -r dataframe_8
-- Which genres have the most forthcoming titles?
SELECT CATEGORY, COUNT(*) AS title_count
FROM ALPINE_PEAKS_PUBLISHING.BOOKS.FORTHCOMING_RELEASES
GROUP BY CATEGORY
ORDER BY title_count DESC;

### Ask CoCo for the view definition. 🤖
Build a view in your own database that gives downstream café-planning models exactly the Cooking titles they need — curated columns, filtered to Cooking, and zero-copy.

🤖 **CoCo Prompt** — *click to copy, then paste into the CoCo panel:*

<pre style="user-select: all; -webkit-user-select: all; background: rgba(41, 181, 232, 0.12); border-left: 4px solid #29B5E8; padding: 12px; border-radius: 0 4px 4px 0; cursor: pointer; white-space: pre-wrap; word-wrap: break-word;">@ALPINE_PEAKS_PUBLISHING Write SQL, but do not run it, that creates a schema MARKETPLACE_LAB in my database, then creates a view CAFE_COOKING_CATALOG in it selecting title, author, and release_date from ALPINE_PEAKS_PUBLISHING.BOOKS.FORTHCOMING_RELEASES filtered to the Cooking category. Explain why a view over a shared table is still zero-copy.</pre>

> :point_right: Read CoCo's SQL, paste it into the cell below, and run it. Fully-qualify the view with your own database.

<details>
<summary>📖 Reference solution</summary>

    CREATE SCHEMA IF NOT EXISTS {{my_db}}.MARKETPLACE_LAB;

    CREATE OR REPLACE VIEW {{my_db}}.MARKETPLACE_LAB.CAFE_COOKING_CATALOG AS
    SELECT TITLE, AUTHOR, RELEASE_DATE
    FROM ALPINE_PEAKS_PUBLISHING.BOOKS.FORTHCOMING_RELEASES
    WHERE CATEGORY = 'Cooking';

A view stores only its definition — no rows — so it reads the provider's live data on every query. Downstream models select from your curated view instead of the raw share.
</details>

In [ ]:
%%sql -r dataframe_9
-- Paste CoCo's view definition on the blank line below — or write your own
-- (create MARKETPLACE_LAB in your DB, then a CAFE_COOKING_CATALOG view over the share)


In [ ]:
%%sql -r dataframe_10
-- Read your curated view — this is what downstream models would query
SELECT * FROM {{my_db}}.MARKETPLACE_LAB.CAFE_COOKING_CATALOG
ORDER BY RELEASE_DATE;

> 🔎 **Review the output.** You built a view over a publisher's live catalog — data you don't own — and not a single row landed in your database. The view stores only its definition and reads the provider's current data every time.

> :bulb: **The engineering connection.** Downstream café-planning models now read `MARKETPLACE_LAB.CAFE_COOKING_CATALOG`, not the raw share. If Alpine Peaks reshapes their schema, you fix one view instead of every model. In real pipelines this is how you wrap supplier feeds and partner data behind a stable, governed interface — without duplicating storage.

## 7 · Operationalize With a Dynamic Table
Your view is a live query — but what if you need a **materialized**, always-fresh subset for downstream models? A **Dynamic Table** does exactly that: you define the SQL once, set a freshness target, and Snowflake handles the rest.

This is the engineering pattern: shared data arrives via zero-copy, you curate it into a Dynamic Table with a `TARGET_LAG`, and consumers query your table — never touching the share directly.

> ⚠️ **Requires Section 6:** This cell uses the `MARKETPLACE_LAB` schema you created above — complete and run Section 6 first, or this will fail with "schema does not exist."

In [ ]:
%%sql -r dataframe_11
CREATE OR REPLACE DYNAMIC TABLE {{my_db}}.MARKETPLACE_LAB.CAFE_COOKING_LIVE
    TARGET_LAG = '1 hour'
    WAREHOUSE  = {{my_wh}}
AS
SELECT
    TITLE,
    AUTHOR,
    CATEGORY,
    RELEASE_DATE,
    DATEDIFF('day', CURRENT_DATE(), RELEASE_DATE) AS days_until_release
FROM ALPINE_PEAKS_PUBLISHING.BOOKS.FORTHCOMING_RELEASES
WHERE CATEGORY = 'Cooking';

> 🔎 **What this gives you.** If Alpine Peaks adds new Cooking titles tomorrow, your `CAFE_COOKING_LIVE` table refreshes automatically within the `TARGET_LAG` window — no pipeline to maintain, no scheduler to configure. The `days_until_release` column is recomputed on each refresh, so downstream dashboards always show accurate countdowns.
>
> This is the difference between the analyst's view (live but re-computed every query) and your Dynamic Table (materialized, scheduled, zero-maintenance). Both are valid — you pick based on freshness needs and query volume.

> :bulb: **FULL vs. incremental refresh.** Because this query uses `CURRENT_DATE()` (a non-deterministic function) and reads from a shared source, the Dynamic Table runs in **FULL refresh** mode — it recomputes the whole result each cycle rather than tracking incremental changes. That's expected and fine for a small catalog; you'll see the mode called out in the `CREATE` output message. For large, append-only sources you'd design the query to qualify for incremental refresh.

## Check Your Work
Run the next cell to confirm your role can reach the shared database. Prints a ✅ / ⛔ for each check — no need to read the code.

In [ ]:
from snowflake.snowpark.context import get_active_session
session = get_active_session()
user = session.get_current_user().strip('"')
my_db = user + '_DB'
checks = [
    ("You can reach the shared Roastery data",
     "SELECT COUNT(*) FROM PLATPLUS_DB.ROASTERY_SALES.STORES", 1),
    ("Your role can query the Alpine Peaks Publishing catalog",
     "SELECT COUNT(*) FROM ALPINE_PEAKS_PUBLISHING.BOOKS.FORTHCOMING_RELEASES", 1),
    ("Your CAFE_COOKING_CATALOG view returns Cooking titles",
     f"SELECT COUNT(*) FROM {my_db}.MARKETPLACE_LAB.CAFE_COOKING_CATALOG", 1),
]
passed = 0
for label, sql, need in checks:
    try:
        n = session.sql(sql).collect()[0][0]
        ok = n >= need
    except Exception:
        n, ok = None, False
    passed += 1 if ok else 0
    icon = '✅' if ok else '⛔'
    detail = f'(found {n})' if n is not None else '(not accessible — re-run the steps above)'
    print(f'{icon}  {label}  {detail}')
print('-' * 52)
if passed == len(checks):
    print(f'🎉 ALL {len(checks)} CHECKS PASSED — lab complete! Nice work.')
else:
    print(f'⚠️  {passed}/{len(checks)} passed — re-run the cells above for any ⛔ item, then run this cell again.')

### Check your understanding. 🧠
A few quick questions on this lab's key themes. Run each cell below, choose an answer, and the feedback appears inline — no grade, just a self-check.

In [ ]:
from resources.knowledgecheck import data_engineer_quiz as quiz
quiz.display_question("q1")

In [ ]:
from resources.knowledgecheck import data_engineer_quiz as quiz
quiz.display_question("q2")

In [ ]:
from resources.knowledgecheck import data_engineer_quiz as quiz
quiz.display_question("q3")

### Key takeaways. ❄️
- **Zero copy** means no data moves — a shared database is a live window onto the provider's data, no storage cost on your side
- The **public Marketplace** is self-serve — search it in Snowsight, or drive it with CoCo's `/marketplace-search` (find listings) and `/get-marketplace-listing-details` (inspect one)
- **Private Listings** are for curated, account-specific data sharing — not visible on the public Marketplace
- Getting access = **`IMPORTED PRIVILEGES`** on the shared database — the one consumer-side lever; it surfaces exactly the objects the provider shared (here, one table). Finer control comes from the **provider** (database roles) or from a **view** you build over the share
- A **view over a shared table** stores no rows — it's a zero-copy, stable interface you give downstream consumers, and it insulates them from provider schema changes
- A **Dynamic Table** is a materialized, auto-refreshing version of a view — set `TARGET_LAG` and Snowflake handles the rest; shared sources + non-deterministic functions like `CURRENT_DATE()` trigger FULL refresh mode instead of incremental
- Decide per dataset: **load it in** (your own source files) vs. **query in place** (partner / reference data)

### Notebook completion. 🏁
You're done — your notebook is saved automatically.

> 💡 **Moving on to another lab?** Shut down this notebook's **kernel** — click **Connected** at the top → **Shut down kernel**. All your lab notebooks share one compute service, so clearing a finished kernel frees up memory for the next lab. Your service stays running, ready in seconds.

> 💡 **Finished for the day?** **Suspend your service** — click **Connected** → hover your service name → **Suspend**. This stops compute; it resumes automatically next time you run a cell.